# 📊 Statistics & Probability for Machine Learning

**Goal**: Build deep intuition from scratch — every concept is connected to real ML/DL behaviour.

**Libraries used**: `numpy`, `scipy`, `matplotlib`, `sklearn`, `torch`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12
print('Libraries loaded!')

---
## 1. Populations & Sampling

Your training dataset is a **sample** from an unknown true distribution $P$.
The model tries to learn $P$ from that sample — this is generalisation.

In [ ]:
# Simulate a 'true' population (e.g., house prices in a city)
np.random.seed(42)
population = np.random.normal(loc=300_000, scale=80_000, size=1_000_000)

# Draw samples of different sizes
sample_sizes = [10, 100, 1000, 10000]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for ax, n in zip(axes, sample_sizes):
    sample = np.random.choice(population, size=n)
    ax.hist(sample, bins=30, density=True, alpha=0.6, color='steelblue')
    ax.axvline(population.mean(), color='red', linestyle='--', label=f'True mean')
    ax.axvline(sample.mean(), color='green', linestyle='-', label=f'Sample mean')
    ax.set_title(f'n = {n}\nSample mean: {sample.mean():.0f}')
    ax.legend(fontsize=8)

plt.suptitle('How Sample Size Affects Estimation of the Population Mean', y=1.02)
plt.tight_layout()
plt.show()

print(f"True population mean: {population.mean():.2f}")
print("\n💡 ML Insight: Larger datasets → better approximation of the true distribution.")
print("   This is why more data almost always helps in ML.")

---
## 2. Mean, Median, Mode & Their Loss Functions

Each measure of centre **minimises a different loss**:

| Estimator | Minimises | ML Loss |
|-----------|-----------|--------|
| Mean | $\sum(x_i - c)^2$ | MSE |
| Median | $\sum|x_i - c|$ | MAE |
| Mode | Misclassification rate | 0-1 loss |

In [ ]:
# Create skewed data with outliers (e.g., incomes)
np.random.seed(0)
data = np.concatenate([np.random.exponential(50_000, 1000), [2_000_000, 5_000_000]])  # outliers

mean_val   = np.mean(data)
median_val = np.median(data)

# Show how each minimises a loss
candidates = np.linspace(0, 300_000, 500)
mse_loss = [np.mean((data - c)**2) for c in candidates]
mae_loss = [np.mean(np.abs(data - c)) for c in candidates]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(candidates, mse_loss, color='steelblue')
axes[0].axvline(mean_val, color='red', linestyle='--', label=f'Mean = {mean_val:.0f}')
axes[0].set_title('MSE Loss — Minimised by the Mean')
axes[0].set_xlabel('Candidate value c')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(candidates, mae_loss, color='darkorange')
axes[1].axvline(median_val, color='green', linestyle='--', label=f'Median = {median_val:.0f}')
axes[1].set_title('MAE Loss — Minimised by the Median')
axes[1].set_xlabel('Candidate value c')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean:   {mean_val:>10.0f}  ← pulled far right by outliers")
print(f"Median: {median_val:>10.0f}  ← robust to outliers")
print("\n💡 Use MAE/Huber loss when your labels have outliers (e.g., house prices, click counts).")

---
## 3. Expected Value & Why Loss Functions Are Expectations

In [ ]:
# Expected value of a dice roll
outcomes = np.array([1, 2, 3, 4, 5, 6])
probs = np.ones(6) / 6
E_X = np.sum(outcomes * probs)
print(f"E[X] for fair die = {E_X}")

# Monte Carlo approximation of E[X]: law of large numbers
rolls = np.random.randint(1, 7, size=10_000)
running_avg = np.cumsum(rolls) / np.arange(1, len(rolls)+1)

plt.figure(figsize=(10, 4))
plt.plot(running_avg, alpha=0.8, color='steelblue', label='Running average of rolls')
plt.axhline(E_X, color='red', linestyle='--', label=f'True E[X] = {E_X}')
plt.xlabel('Number of rolls')
plt.ylabel('Average')
plt.title('Law of Large Numbers: Sample Average → Expected Value')
plt.legend()
plt.xscale('log')
plt.show()

print("\n💡 ML batch loss: (1/B) Σ loss_i  is a Monte Carlo estimate of E[loss]")
print("   Larger batch size → better estimate → less gradient noise.")
print("   But smaller batches add beneficial noise that can escape sharp minima!")

---
## 4. Variance, Std Dev & Batch Normalisation

In [ ]:
np.random.seed(1)

# Simulate neuron activations in a deep network layer
activations = np.random.normal(loc=5, scale=3, size=1000)

# Batch Normalisation: subtract mean, divide by std
mu = activations.mean()
sigma = activations.std()
normalized = (activations - mu) / (sigma + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(activations, bins=40, color='steelblue', alpha=0.7)
axes[0].set_title(f'Raw Activations\nμ={mu:.2f}, σ={sigma:.2f}')
axes[0].set_xlabel('Activation value')

axes[1].hist(normalized, bins=40, color='darkorange', alpha=0.7)
axes[1].set_title(f'After Batch Norm\nμ={normalized.mean():.4f}, σ={normalized.std():.4f}')
axes[1].set_xlabel('Normalised activation')

plt.tight_layout()
plt.show()

print("💡 Batch Norm formula: x̂ = (x - μ_B) / sqrt(σ²_B + ε)")
print("   This stabilises the input distribution to each layer during training.")
print("   Without it, shifting distributions through layers slow training (internal covariate shift).")

# Bessel's correction demo
n = 10
sample = np.random.normal(0, 1, n)
biased_var   = np.sum((sample - sample.mean())**2) / n
unbiased_var = np.sum((sample - sample.mean())**2) / (n - 1)
print(f"\nBiased variance (÷N):   {biased_var:.4f}")
print(f"Unbiased variance (÷N-1): {unbiased_var:.4f}")
print("True variance: 1.0  ← Bessel's correction gives a better estimate")

---
## 5. Covariance, Correlation & the Covariance Matrix

In [ ]:
np.random.seed(42)

# Generate correlated features
n = 300
x1 = np.random.normal(0, 1, n)
x2 =  0.9 * x1 + 0.1 * np.random.normal(0, 1, n)  # highly correlated with x1
x3 = -0.7 * x1 + 0.5 * np.random.normal(0, 1, n)  # negatively correlated
x4 = np.random.normal(0, 1, n)                      # independent

X = np.column_stack([x1, x2, x3, x4])
cov = np.cov(X.T)  # 4x4 covariance matrix

# Correlation matrix (normalised covariance)
D = np.diag(1 / np.sqrt(np.diag(cov)))
corr = D @ cov @ D

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im1 = axes[0].imshow(cov, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Covariance Matrix')
axes[0].set_xticks(range(4)); axes[0].set_yticks(range(4))
axes[0].set_xticklabels(['x1','x2','x3','x4']); axes[0].set_yticklabels(['x1','x2','x3','x4'])
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('Correlation Matrix (Normalised)')
axes[1].set_xticks(range(4)); axes[1].set_yticks(range(4))
axes[1].set_xticklabels(['x1','x2','x3','x4']); axes[1].set_yticklabels(['x1','x2','x3','x4'])
plt.colorbar(im2, ax=axes[1])
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("💡 In PCA, we diagonalise this covariance matrix.")
print("   The eigenvectors give directions of max variance (principal components).")

---
## 6. Probability Distributions — The Full Picture

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
x = np.linspace(-4, 4, 300)

# 1. Gaussian
for mu, sigma, c in [(0, 1, 'blue'), (0, 2, 'red'), (1, 0.5, 'green')]:
    axes[0,0].plot(x, stats.norm.pdf(x, mu, sigma), label=f'μ={mu}, σ={sigma}', color=c)
axes[0,0].set_title('Normal Distribution N(μ, σ²)')
axes[0,0].legend()

# 2. Bernoulli
for p, c in [(0.3, 'blue'), (0.5, 'red'), (0.8, 'green')]:
    axes[0,1].bar([0,1], [1-p, p], alpha=0.5, color=c, width=0.2,
                  align='center', label=f'p={p}')
axes[0,1].set_title('Bernoulli — Binary Outcomes')
axes[0,1].set_xticks([0,1]); axes[0,1].set_xticklabels(['0 (neg)','1 (pos)'])
axes[0,1].legend()

# 3. Binomial
k = np.arange(0, 31)
for n_trial, p, c in [(30, 0.3, 'blue'), (30, 0.5, 'red'), (30, 0.8, 'green')]:
    axes[0,2].plot(k, stats.binom.pmf(k, n_trial, p), 'o-', color=c, label=f'n={n_trial},p={p}')
axes[0,2].set_title('Binomial — Count of Successes in n Trials')
axes[0,2].legend()

# 4. Uniform
xu = np.linspace(-0.5, 3.5, 300)
axes[1,0].plot(xu, stats.uniform.pdf(xu, 0, 3), label='U(0,3)', color='blue', lw=2)
axes[1,0].fill_between(xu, stats.uniform.pdf(xu, 0, 3), alpha=0.3)
axes[1,0].set_title('Uniform Distribution — Equal Probability')
axes[1,0].legend()

# 5. Softmax (categorical distribution)
logits = np.array([2.0, 1.0, 0.1, -1.0, -2.0])
probs = np.exp(logits) / np.exp(logits).sum()
axes[1,1].bar(range(5), probs, color='purple', alpha=0.7)
axes[1,1].set_title('Categorical (after Softmax)\nNetwork output for 5-class problem')
axes[1,1].set_xticks(range(5))
axes[1,1].set_xticklabels([f'Class {i}\nlogit={l:.1f}' for i,l in enumerate(logits)], fontsize=9)
for i, p in enumerate(probs):
    axes[1,1].text(i, p+0.01, f'{p:.2%}', ha='center')

# 6. Temperature scaling
temps = [0.1, 1.0, 5.0]
for T, c in zip(temps, ['blue','red','green']):
    p = np.exp(logits/T) / np.exp(logits/T).sum()
    axes[1,2].plot(range(5), p, 'o-', label=f'T={T}', color=c)
axes[1,2].set_title('Temperature Scaling of Softmax\nT→0: argmax, T→∞: uniform')
axes[1,2].set_xticks(range(5)); axes[1,2].set_xticklabels([f'C{i}' for i in range(5)])
axes[1,2].legend()

plt.tight_layout()
plt.show()

print("💡 Temperature T controls the 'confidence' of the softmax.")
print("   Used in knowledge distillation and language model sampling (nucleus/top-p).")

---
## 7. Central Limit Theorem — Why Gaussian is Everywhere

In [ ]:
np.random.seed(0)

# Start with a very non-normal distribution (exponential)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

sample_sizes = [1, 2, 5, 30]
n_experiments = 10_000

for col, n in enumerate(sample_sizes):
    # Row 0: Exponential distribution
    sample_means = [np.random.exponential(1, n).mean() for _ in range(n_experiments)]
    axes[0, col].hist(sample_means, bins=50, density=True, color='steelblue', alpha=0.7)
    # Overlay the predicted normal
    mean_pred = 1.0  # E[Exp(1)] = 1
    std_pred  = 1.0 / np.sqrt(n)  # std of sample mean
    xrange = np.linspace(min(sample_means), max(sample_means), 200)
    axes[0, col].plot(xrange, stats.norm.pdf(xrange, mean_pred, std_pred), 'r-', lw=2)
    axes[0, col].set_title(f'Exp(1), n={n}')
    axes[0, col].set_xlabel('Sample mean')
    if col == 0: axes[0, col].set_ylabel('Density')

    # Row 1: Uniform distribution
    sample_means = [np.random.uniform(0, 1, n).mean() for _ in range(n_experiments)]
    axes[1, col].hist(sample_means, bins=50, density=True, color='darkorange', alpha=0.7)
    mean_pred = 0.5
    std_pred  = (1/np.sqrt(12)) / np.sqrt(n)
    xrange = np.linspace(min(sample_means), max(sample_means), 200)
    axes[1, col].plot(xrange, stats.norm.pdf(xrange, mean_pred, std_pred), 'r-', lw=2)
    axes[1, col].set_title(f'Uniform(0,1), n={n}')
    axes[1, col].set_xlabel('Sample mean')
    if col == 0: axes[1, col].set_ylabel('Density')

plt.suptitle('Central Limit Theorem: Sample Means Converge to Gaussian (red curve) Regardless of Source Distribution', y=1.02)
plt.tight_layout()
plt.show()

print("💡 SGD gradient = average of per-sample gradients in a mini-batch")
print("   By CLT, this gradient estimate is approximately Gaussian distributed.")
print("   This Gaussian noise actually helps escape sharp minima → implicit regularisation!")

---
## 8. Bayes' Theorem — From Spam Filters to Neural Priors

In [ ]:
# ── Naïve Bayes Email Classifier from Scratch ──

# Imagine we have two classes: spam (1) and ham (0)
# Words: 'money', 'free', 'meeting', 'project'
# We count how often each word appears per class

words = ['money', 'free', 'meeting', 'project']

# P(word | class) — how likely each word is in spam vs ham
p_word_given_spam = {'money': 0.8, 'free': 0.7, 'meeting': 0.1, 'project': 0.05}
p_word_given_ham  = {'money': 0.1, 'free': 0.1, 'meeting': 0.6, 'project': 0.5}

# Prior: 30% of emails are spam
p_spam = 0.3
p_ham  = 0.7

# New email contains: 'money', 'free'
email = ['money', 'free']

# Compute likelihoods (Naïve assumption: words are independent)
likelihood_spam = p_spam
likelihood_ham  = p_ham

for word in email:
    likelihood_spam *= p_word_given_spam[word]
    likelihood_ham  *= p_word_given_ham[word]

# Unnormalised posterior (proportional to P(class|email))
evidence = likelihood_spam + likelihood_ham
p_spam_given_email = likelihood_spam / evidence
p_ham_given_email  = likelihood_ham  / evidence

print("=== Naïve Bayes Spam Classifier ===")
print(f"Email contains: {email}")
print(f"\nPrior P(spam) = {p_spam}")
print(f"Likelihood P(email|spam) ∝ {likelihood_spam:.6f}")
print(f"Likelihood P(email|ham)  ∝ {likelihood_ham:.6f}")
print(f"\nPosterior P(spam|email) = {p_spam_given_email:.2%}")
print(f"Posterior P(ham|email)  = {p_ham_given_email:.2%}")
print(f"\nDecision: {'SPAM 🚫' if p_spam_given_email > 0.5 else 'HAM ✅'}")

# ── Bayesian interpretation of regularisation ──
print("\n" + "="*50)
print("Bayesian Interpretation of Regularisation:")
print("  L2 regularisation = Gaussian prior on weights: P(w) = N(0, 1/λ)")
print("  L1 regularisation = Laplace prior on weights:  P(w) = Laplace(0, 1/λ)")
print("  MAP estimation = maximise P(w|data) = P(data|w) * P(w)")
print("  This is equivalent to adding a penalty to the loss function!")

In [ ]:
# Visualise L2 (Gaussian) vs L1 (Laplace) priors
w = np.linspace(-3, 3, 300)
l2_prior = stats.norm.pdf(w, 0, 1)      # Gaussian
l1_prior = stats.laplace.pdf(w, 0, 1)   # Laplace

plt.figure(figsize=(8, 4))
plt.plot(w, l2_prior, 'b-', lw=2, label='Gaussian prior → L2 (Ridge)')
plt.plot(w, l1_prior, 'r-', lw=2, label='Laplace prior → L1 (Lasso)')
plt.fill_between(w, l1_prior, alpha=0.15, color='red')
plt.fill_between(w, l2_prior, alpha=0.15, color='blue')
plt.xlabel('Weight value w')
plt.ylabel('Prior probability P(w)')
plt.title('Priors Encode Our Belief About Weight Values\n(Laplace has sharper peak → encourages sparsity → L1)')
plt.legend()
plt.show()

print("💡 Laplace prior has a sharp peak at 0 → strongly encourages weights to be EXACTLY zero")
print("   This is why L1 regularisation gives sparse models (feature selection!)")

---
## 9. Maximum Likelihood Estimation — Where Loss Functions Come From

In [ ]:
# ── MLE: Fitting a Gaussian to data from scratch ──
np.random.seed(7)
true_mu, true_sigma = 3.0, 1.5
data = np.random.normal(true_mu, true_sigma, 200)

# Log-likelihood as a function of μ (fixing σ at true value for visualisation)
mu_candidates = np.linspace(0, 6, 200)
log_likelihoods = [
    np.sum(stats.norm.logpdf(data, mu, true_sigma))
    for mu in mu_candidates
]

# Analytical MLE solution
mle_mu    = data.mean()
mle_sigma = data.std()  # biased, but MLE

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(mu_candidates, log_likelihoods, 'steelblue', lw=2)
axes[0].axvline(mle_mu, color='red', linestyle='--', label=f'MLE μ̂ = {mle_mu:.3f}')
axes[0].axvline(true_mu, color='green', linestyle=':', label=f'True μ = {true_mu}')
axes[0].set_xlabel('μ (candidate mean)')
axes[0].set_ylabel('Log-Likelihood ℓ(μ)')
axes[0].set_title('MLE: Find μ that Maximises Log-Likelihood')
axes[0].legend()

axes[1].hist(data, bins=30, density=True, alpha=0.5, color='steelblue', label='Data')
x_range = np.linspace(data.min()-1, data.max()+1, 300)
axes[1].plot(x_range, stats.norm.pdf(x_range, mle_mu, mle_sigma), 'r-', lw=2, label='MLE fit')
axes[1].plot(x_range, stats.norm.pdf(x_range, true_mu, true_sigma), 'g--', lw=2, label='True distribution')
axes[1].set_title('MLE Fitted Distribution vs Truth')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"MLE estimate: μ̂={mle_mu:.3f}, σ̂={mle_sigma:.3f}")
print(f"True params:  μ={true_mu},   σ={true_sigma}")

In [ ]:
# ── MLE → Loss Functions — Derive them yourself! ──

print("=" * 60)
print("DERIVATION: MSE Loss from Gaussian Likelihood")
print("=" * 60)
print("""
Assume: y_i = f(x_i; θ) + ε_i,  ε_i ~ N(0, σ²)

Likelihood:  L(θ) = Π_i (1/√(2πσ²)) exp(-(y_i - f(x_i;θ))² / 2σ²)

Log-likelihood:
  ℓ(θ) = -N/2 log(2πσ²) - 1/(2σ²) Σ_i (y_i - f(x_i;θ))²

Maximise ℓ(θ)  ⟺  Minimise Σ_i (y_i - f(x_i;θ))²
                ⟺  Minimise MSE  ✓
""")

print("=" * 60)
print("DERIVATION: Binary Cross-Entropy from Bernoulli Likelihood")
print("=" * 60)
print("""
Assume: y_i ~ Bernoulli(σ(f(x_i; θ)))

Likelihood:  L(θ) = Π_i p_i^y_i * (1-p_i)^(1-y_i)
             where p_i = sigmoid(f(x_i; θ))

Log-likelihood:
  ℓ(θ) = Σ_i [y_i log(p_i) + (1-y_i) log(1-p_i)]

Maximise ℓ(θ)  ⟺  Minimise -ℓ(θ) = Binary Cross-Entropy Loss  ✓
""")

---
## 10. Linear Regression — From Scratch with Normal Equations & Gradient Descent

In [ ]:
np.random.seed(42)

# Generate data: y = 2x + 1 + noise
X_raw = np.random.uniform(0, 10, 100)
y = 2 * X_raw + 1 + np.random.normal(0, 2, 100)

# Design matrix with bias column
X = np.column_stack([np.ones(len(X_raw)), X_raw])  # shape (100, 2)

# ── Method 1: Normal Equation (closed-form) ──
w_normal = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"Normal Equation: w0 (bias)={w_normal[0]:.3f}, w1 (slope)={w_normal[1]:.3f}")

# ── Method 2: Gradient Descent from scratch ──
def mse(X, y, w):
    return np.mean((X @ w - y)**2)

def gradient(X, y, w):
    N = len(y)
    return (2/N) * X.T @ (X @ w - y)

w_gd = np.zeros(2)
lr = 0.005
losses = []

for epoch in range(500):
    losses.append(mse(X, y, w_gd))
    w_gd -= lr * gradient(X, y, w_gd)

print(f"Gradient Descent: w0 (bias)={w_gd[0]:.3f}, w1 (slope)={w_gd[1]:.3f}")
print(f"True values:      w0=1.000, w1=2.000")

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x_line = np.linspace(0, 10, 100)
axes[0].scatter(X_raw, y, alpha=0.5, label='Data')
axes[0].plot(x_line, w_normal[0] + w_normal[1]*x_line, 'r-', lw=2, label='Normal Eq.')
axes[0].plot(x_line, w_gd[0] + w_gd[1]*x_line, 'g--', lw=2, label='Gradient Descent')
axes[0].set_title('Linear Regression Fit')
axes[0].legend()

axes[1].plot(losses, color='steelblue')
axes[1].set_title('Training Loss (MSE) over Iterations')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# ── Bias-Variance Tradeoff Visualisation ──
from numpy.polynomial.polynomial import polyfit

np.random.seed(3)
x_true = np.linspace(0, 1, 200)
y_true = np.sin(2 * np.pi * x_true)

degrees = [1, 3, 9]
n_train = 15

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, d in zip(axes, degrees):
    train_fits = []
    for _ in range(50):
        x_train = np.random.uniform(0, 1, n_train)
        y_train = np.sin(2 * np.pi * x_train) + np.random.normal(0, 0.2, n_train)
        coeffs = np.polyfit(x_train, y_train, d)
        y_fit = np.polyval(coeffs, x_true)
        train_fits.append(y_fit)
        ax.plot(x_true, y_fit, 'b-', alpha=0.1)
    
    mean_fit = np.mean(train_fits, axis=0)
    ax.plot(x_true, y_true, 'r-', lw=2, label='True f(x)')
    ax.plot(x_true, mean_fit, 'b-', lw=2, label='Mean prediction')
    ax.set_title(f'Degree {d} Polynomial\n{"Underfit (High Bias)" if d==1 else "Good Fit" if d==3 else "Overfit (High Variance)"}')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)

plt.suptitle('Bias-Variance Tradeoff: Each blue line = one model trained on different data sample')
plt.tight_layout()
plt.show()

print("💡 Degree 1: All blue lines close together (low variance), but far from truth (high bias)")
print("   Degree 9: Lines spread wildly (high variance) → overfitting")
print("   Solution: Regularisation, cross-validation, more data")

---
## 11. Logistic Regression — From Scratch

In [ ]:
# Generate linearly separable 2D data
np.random.seed(42)
X_cl, y_cl = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                  n_informative=2, random_state=42, n_clusters_per_class=1)

# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Binary cross-entropy loss
def bce_loss(y_true, y_pred):
    eps = 1e-9
    return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

# Add bias column
X_b = np.column_stack([np.ones(len(X_cl)), X_cl])

# From-scratch logistic regression via gradient descent
w = np.zeros(3)
lr = 0.1
losses_lr = []

for epoch in range(300):
    z = X_b @ w
    p = sigmoid(z)
    loss = bce_loss(y_cl, p)
    losses_lr.append(loss)
    # Gradient of BCE w.r.t. w: X^T (p - y) / N
    grad = X_b.T @ (p - y_cl) / len(y_cl)
    w -= lr * grad

print(f"Final Loss: {losses_lr[-1]:.4f}")
preds = (sigmoid(X_b @ w) > 0.5).astype(int)
accuracy = np.mean(preds == y_cl)
print(f"Training Accuracy: {accuracy:.2%}")

# Plot decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(X_cl[:,0].min()-1, X_cl[:,0].max()+1, 200),
                     np.linspace(X_cl[:,1].min()-1, X_cl[:,1].max()+1, 200))
grid = np.column_stack([np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()])
zz = sigmoid(grid @ w).reshape(xx.shape)

axes[0].contourf(xx, yy, zz, levels=50, cmap='RdBu_r', alpha=0.6)
axes[0].contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
axes[0].scatter(X_cl[:,0], X_cl[:,1], c=y_cl, cmap='RdBu_r', edgecolors='k', s=40)
axes[0].set_title(f'Logistic Regression Decision Boundary\nAccuracy: {accuracy:.2%}')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')

axes[1].plot(losses_lr, color='darkorange', lw=2)
axes[1].set_title('Binary Cross-Entropy Loss During Training')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

plt.tight_layout()
plt.show()

print("\n💡 This is EXACTLY what a 1-layer neural network with sigmoid activation does!")
print("   A deep network = many logistic regressions stacked and composed.")

---
## 12. Putting It All Together: Probabilistic View of a Neural Network

In [ ]:
import torch
import torch.nn as nn

# A simple 2-layer network for binary classification
# Every component has a probabilistic meaning

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()   # Output = P(y=1 | x)  ← Bernoulli probability
        )
    def forward(self, x):
        return self.net(x).squeeze()

# Data
X_t = torch.tensor(X_cl, dtype=torch.float32)
y_t = torch.tensor(y_cl, dtype=torch.float32)

model = SimpleNet()
# BCELoss = -1/N Σ [y log(p) + (1-y) log(1-p)]
# = Negative log-likelihood under Bernoulli distribution  ← MLE!
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

torch_losses = []
for epoch in range(300):
    optimizer.zero_grad()
    probs = model(X_t)
    loss = criterion(probs, y_t)
    loss.backward()
    optimizer.step()
    torch_losses.append(loss.item())

with torch.no_grad():
    preds_t = (model(X_t) > 0.5).float()
    acc = (preds_t == y_t).float().mean().item()

print(f"PyTorch Neural Net Accuracy: {acc:.2%}")
plt.figure(figsize=(8, 4))
plt.plot(torch_losses, color='purple', lw=2)
plt.title('Neural Network Training Loss (BCELoss = Negative Log-Likelihood)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.show()

print("\n🎯 SUMMARY: Everything connects!")
print("  Gaussian noise assumption → MSE loss (regression)")
print("  Bernoulli likelihood → Binary Cross-Entropy (binary classification)")
print("  Categorical likelihood → Cross-Entropy (multi-class)")
print("  Gaussian prior on weights → L2 regularisation")
print("  Laplace prior on weights → L1 regularisation")
print("  Gradient descent → MLE parameter estimation")